# Retrieval Benchmark — HeLa & Brodae

Evaluates the trained dual-encoder's retrieval ability on Gabriel's HeLa and Brodae datasets, then runs the same FDR-controlled experiments (top-1 precision + Target-Decoy) defined in `src/retrieval/search.py`.

**Inputs expected under `DATA_ROOT`** (matches `Data - Copy/`):
- `helaqc_PSM.csv` / `sbrodae_PSM.csv` — ground-truth PSMs (Sequest + Percolator q-values).
- `Hela trypsin digest.csv` / `brodae trypsin digest.csv` — target peptide DB.
- `human_extended_normalized.fasta` / `proteins_decoy_brodae.fasta` — decoy DB.
- One or more `.mzML` / `.mgf` files matching the PSM `Spectrum File` column.
- `percolator.exe` (optional).

**On GCP:** clone this repo into the VM, push `Data - Copy/` and your mzML/MGF files to a GCS bucket, `gsutil cp` them onto the VM, then point `DATA_ROOT` and `MS_FILES` at the local paths.

## 1. Environment

In [ ]:
from __future__ import annotations

import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Repo + vendored InstaNovo on sys.path (notebook lives in notebooks/).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in [REPO_ROOT, REPO_ROOT / "InstaNovo"]:
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repo : {REPO_ROOT}")
print(f"Device: {DEVICE}")

## 2. Pick a dataset and point at the data

Switch between local (`Data - Copy`) and the GCP VM by changing **only** `DATA_ROOT` and `MS_FILES`.

In [ ]:
from src.retrieval.benchmarks import HELA_PRESET, BRODAE_PRESET

# ── EDIT HERE ────────────────────────────────────────────────
DATA_ROOT = REPO_ROOT / "Data - Copy"          # local
# DATA_ROOT = Path("/home/jupyter/Data")        # GCP VM

DATASET = "brodae"                             # "hela" or "brodae"

MS_FILES = sorted(DATA_ROOT.glob("*.mzML")) + sorted(DATA_ROOT.glob("*.mgf"))
# MS_FILES = [Path("/home/jupyter/mzml/20230608_EV_EVO_KK_FAIMS_2CV_Endurance_30SPD_1475_Sbrodae.mzML")]

preset = {"hela": HELA_PRESET, "brodae": BRODAE_PRESET}[DATASET]
bench_cfg = preset(DATA_ROOT, ms_files=MS_FILES)
print(bench_cfg)

## 3. Load InstaNovo backbone + dual encoders

In [ ]:
import yaml
from instanovo.transformer.model import InstaNovo
from instanovo.utils.residues import ResidueSet
from instanovo.constants import LEGACY_PTM_TO_UNIMOD

from src.models.insta_search_spectrum_encoder import InstaSearchSpectrumEncoder
from src.models.peptide_encoder import PeptideEncoder
from src.utils.config import D_MODEL, N_HEADS, D_FF, N_LAYERS, EMBED_DIM, DROPOUT

INSTANOVO_CHECKPOINT = "instanovo-v1.2.0"

_orig_load = torch.load
torch.load = lambda *a, **kw: _orig_load(*a, **{**kw, "weights_only": False})
try:
    instanovo_model, instanovo_config = InstaNovo.from_pretrained(INSTANOVO_CHECKPOINT)
finally:
    torch.load = _orig_load
d_instanovo = int(instanovo_config["dim_model"])

residue_cfg_path = REPO_ROOT / "InstaNovo" / "instanovo" / "configs" / "residues" / "default.yaml"
with open(residue_cfg_path) as f:
    residue_cfg = yaml.safe_load(f)
residue_set = ResidueSet(residue_masses=residue_cfg["residues"], residue_remapping=LEGACY_PTM_TO_UNIMOD)
NUM_AA = len(residue_set.vocab)
print(f"ResidueSet vocab: {NUM_AA}")

model_spec = InstaSearchSpectrumEncoder(
    instanovo_model=instanovo_model, d_instanovo=d_instanovo,
    embed_dim=EMBED_DIM, freeze_encoder=True, dropout=DROPOUT,
).to(DEVICE)
model_pep = PeptideEncoder(
    d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, n_layers=N_LAYERS,
    embed_dim=EMBED_DIM, num_aa=NUM_AA,
).to(DEVICE)

# Load fine-tuned checkpoints — edit paths if needed.
SPEC_CKPT = REPO_ROOT / "checkpoints" / "model_spec.pt"
PEP_CKPT  = REPO_ROOT / "checkpoints" / "model_pep.pt"
if SPEC_CKPT.exists():
    model_spec.load_state_dict(torch.load(SPEC_CKPT, map_location=DEVICE))
    print(f"Loaded {SPEC_CKPT.name}")
if PEP_CKPT.exists():
    model_pep.load_state_dict(torch.load(PEP_CKPT, map_location=DEVICE))
    print(f"Loaded {PEP_CKPT.name}")
model_spec.eval(); model_pep.eval()

## 4. Stage-1 retrieval benchmark

Loads PSMs → targets + decoys → joins to spectra → encodes → HNSW → Recall@k.

In [ ]:
from src.retrieval import HNSWConfig
from src.retrieval.benchmarks import run_retrieval_benchmark

hnsw_cfg = HNSWConfig(embed_dim=EMBED_DIM, M=32, ef_construction=200, ef_search=128, k_retrieve=100)

report, index, db_seqs, is_decoy_db, kept_df, dataset = run_retrieval_benchmark(
    bench_cfg=bench_cfg,
    model_spec=model_spec, model_pep=model_pep,
    residue_set=residue_set,
    hnsw_cfg=hnsw_cfg,
    device=DEVICE,
)

## 5. FDR experiments (Gabriel-style)

Three FDR estimates:
1. **Top-1 precision FDR** over a unique-modified-sequence DB (`compute_fdr`).
2. **Target-Decoy FDR with internal token-level decoys** (`compute_tda_fdr`, reverse-inner).
3. **Target-Decoy FDR with Gabriel's external decoy FASTA** (`compute_external_tda_fdr`).

In [ ]:
from src.retrieval.benchmarks import run_fdr_benchmark

fdr_report = run_fdr_benchmark(
    bench_cfg=bench_cfg,
    model_spec=model_spec, model_pep=model_pep,
    dataset=dataset,
    modified_sequences=kept_df["modified_sequence"].tolist(),
    device=DEVICE,
    use_external_decoys=True,
    db_seqs=db_seqs, is_decoy_db=is_decoy_db,
)

## 6. (Optional) Neural rescoring with InstaNovo decoder

In [ ]:
from src.retrieval.benchmarks import rescore_benchmark

rescorer_model = instanovo_model.to(DEVICE).eval()
rescorer_model.residue_set.update_remapping(LEGACY_PTM_TO_UNIMOD)

report = rescore_benchmark(
    bench_cfg=bench_cfg,
    retrieval=report,
    rescorer=rescorer_model,
    dataset=dataset,
    kept_df=kept_df,
)

## 7. (Optional) Percolator export

Writes `df_percolator_<dataset>.tsv` in the format Gabriel's `df_percolator_sbrodae.tsv` uses, then runs `percolator.exe`.

In [ ]:
from src.retrieval.benchmarks import load_target_peptides, export_percolator_tsv, run_percolator

_, peptide_to_proteins = load_target_peptides(bench_cfg.digest_csv, fixed_mods=bench_cfg.fixed_mods)
perc_tsv = bench_cfg.out_dir / f"df_percolator_{bench_cfg.name}.tsv"

export_percolator_tsv(
    retrieval=report, kept_df=kept_df,
    db_seqs=db_seqs, is_decoy_db=is_decoy_db,
    peptide_to_proteins=peptide_to_proteins,
    out_tsv=perc_tsv,
)

if bench_cfg.percolator_exe and bench_cfg.percolator_exe.exists():
    perc_out = run_percolator(perc_tsv, bench_cfg.percolator_exe, bench_cfg.out_dir)
    print(perc_out)
else:
    print(f"percolator.exe not found at {bench_cfg.percolator_exe} — skipping execution")

## 8. Plots

In [ ]:
import matplotlib.pyplot as plt

out = bench_cfg.out_dir; out.mkdir(parents=True, exist_ok=True)
k_axis = np.arange(1, hnsw_cfg.k_retrieve + 1)
s1 = [(report.stage1_ranks <= k).mean() for k in k_axis]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_axis, s1, lw=2, label="Stage-1 (HNSW)")
if report.rescored_ranks is not None:
    rs = [(report.rescored_ranks <= k).mean() for k in k_axis]
    ax.plot(k_axis, rs, lw=2, ls="--", label="Rescored")
ax.set_xlabel("k"); ax.set_ylabel("Recall@k")
ax.set_title(f"Recall — {bench_cfg.name}"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(out / "recall_curve.png", dpi=150); plt.show()

tda_ext = fdr_report.tda_fdr.get("external")
if tda_ext is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    order = np.argsort(-tda_ext["top1_scores"])
    ax.plot(tda_ext["top1_scores"][order], tda_ext["qvalues"][order], lw=1.5)
    ax.set_xlabel("Top-1 cosine score"); ax.set_ylabel("q-value (external decoys)")
    ax.set_title(f"TDA FDR — {bench_cfg.name}"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out / "tda_fdr.png", dpi=150); plt.show()